# K-fold CV Training (Colab / Local)

Cross-validates `configs/efficientnet_b0_cv.yaml` across 5 folds, then retrains one final
model on the full non-test pool using the epoch count the sweep validated. Standalone -
doesn't touch or depend on `03_train_colab.ipynb`; run this instead of `03_train_colab.ipynb`
when you want a CV-validated model rather than a single 70/15/15 split.

**Kaggle auth** (only needed if `data/` isn't already present): tries, in order, an
existing `KAGGLE_API_TOKEN` env var, `~/.kaggle/access_token`, `~/.kaggle/kaggle.json`,
a Colab secret named `KAGGLE_API_TOKEN` (browser UI only), then an interactive prompt.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Works around a known Windows conda/pip OpenMP DLL conflict (harmless elsewhere).
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

GIT_URL = 'https://github.com/hagairavid18/beilinson.git'
GIT_BRANCH = 'main'


def _run_git(*args: str) -> None:
    """Run git and, on failure, print its actual stdout/stderr before raising - a bare
    CalledProcessError shows the command but not *why* git refused, which is the part
    that actually matters for diagnosing/fixing it. Kept inline (not in training_utils.py)
    because it has to run before we can trust anything on disk, including that module."""
    result = subprocess.run(['git', *args], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"git {' '.join(args)} failed (exit {result.returncode}) - see output above")


try:
    import google.colab  # noqa: F401
    PROJECT_ROOT = Path('/content/beilinson')
    if PROJECT_ROOT.exists():
        try:
            _run_git('-C', str(PROJECT_ROOT), 'pull')
        except RuntimeError:
            # Most likely cause: this clone has local/diverged history (e.g. a commit made
            # directly in a previous Colab session) that a plain pull can't fast-forward.
            # This VM is ephemeral and origin is the source of truth, so reset tracked files
            # to match it - untracked files (checkpoints, the image cache; both under the
            # gitignored artifacts/) are left alone.
            print('git pull failed; resetting tracked files to match origin/%s...' % GIT_BRANCH)
            _run_git('-C', str(PROJECT_ROOT), 'fetch', 'origin', GIT_BRANCH)
            _run_git('-C', str(PROJECT_ROOT), 'reset', '--hard', f'origin/{GIT_BRANCH}')
    else:
        _run_git('clone', '--branch', GIT_BRANCH, GIT_URL, str(PROJECT_ROOT))
except ImportError:
    # Not on Colab (e.g. a local kernel) - use the repo checkout we're already in.
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
(PROJECT_ROOT / 'data').mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Has data already:', any((PROJECT_ROOT / 'data').glob('*/*')))

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [ ]:
import importlib
import sys

import yaml

# Force-reload our own modules (not third-party ones) in dependency order, so re-running
# this cell after a `git pull` picks up the latest code even in an already-running kernel -
# a plain `import`/`from X import Y` is a no-op once a module is already in sys.modules.
for _module_name in ['data_splitter', 'model', 'pytorch_lightning', 'dataset', 'training_utils']:
    if _module_name in sys.modules:
        importlib.reload(sys.modules[_module_name])
    else:
        importlib.import_module(_module_name)

from training_utils import ensure_dataset, run_final_training, run_kfold_sweep

## Config

`CONFIG_PATH` is the CV config - `n_folds`, `test_frac`, and `augment` live in
`configs/efficientnet_b0_cv.yaml`. Point it at a different config (with `n_folds` set)
to cross-validate a different architecture.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'efficientnet_b0_cv.yaml'

with open(CONFIG_PATH, 'r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

print('Config:', CONFIG_PATH)
print(
    f"backbone={CONFIG['model']['backbone']}  n_folds={CONFIG['data']['n_folds']}  "
    f"max_epochs={CONFIG['training']['max_epochs']}  augment={CONFIG['data'].get('augment', False)}"
)

## Dataset

Downloads from Kaggle only if `data/` is empty (see `ensure_dataset` in `training_utils.py`
for the auth fallback chain).

In [ ]:
class_names = ensure_dataset(PROJECT_ROOT)
print(f'{len(class_names)} classes:', class_names)

## K-fold sweep

Trains and evaluates all `data.n_folds` folds back-to-back, then reports mean±std
val_acc across folds - useful signal on a dataset this small, where a single fixed val
split is noisy. Lightning's `Trainer.fit` only trains one model on one split; there's no
built-in k-fold orchestrator, so `training_utils.run_kfold_sweep` just injects a
different `fold_index` into an in-memory copy of `CONFIG_PATH` and runs the full
training pipeline once per fold - no need for a separate yaml file per fold. Each fold
gets its own `artifacts/<exp_name>_fold<i>/` folder (checkpoints, TensorBoard/CSV logs),
and every fold shares the exact same held-out test set (same seed).

Runs the full training budget `n_folds`x, so budget your Colab session accordingly.

In [ ]:
fold_results_df = run_kfold_sweep(CONFIG_PATH, PROJECT_ROOT)
fold_results_df

## Final retrain on the full set

Cross-validation's job is done once it's given a reliable performance estimate and a
trustworthy epoch count - the final model should be trained on *all* the non-test data
(train+val combined), not just the winning fold, so none of the sweep's held-out-val
data goes to waste.

Uses the mean of each fold's best epoch as the epoch budget for this run - with no val
split, there's nothing to early-stop against, so `run_final_training` just trains for
exactly that many epochs and keeps the last epoch's weights (see `data.full_retrain` in
`training_utils._run_training`).

`target_exp_name='efficientnet_b0'` makes the result land in the same
`artifacts/efficientnet_b0/` folder `configs/base.yaml` already points to - so once this
finishes, it *is* the default checkpoint `05_inference.ipynb` and `03_train_colab.ipynb`
discover, with no changes needed anywhere else. Same held-out test set as every fold
(same seed), so `05_inference.ipynb`'s test-set evaluation stays valid against it.

In [ ]:
final_max_epochs = int(round(fold_results_df['best_epoch'].mean()))
print(f'Training final model on the full set for {final_max_epochs} epochs (mean best epoch across folds)')

final_result = run_final_training(
    CONFIG_PATH, PROJECT_ROOT, max_epochs=final_max_epochs, target_exp_name='efficientnet_b0',
)
final_result